# 5장. 드리프트와 이상 감지, AI QA 전략

이 notebook은 1장부터 4장까지의 증거를 하나의 release 판단으로 연결하는 최종 실습입니다. 입력 분포 변화, score와 prediction 분포 변화, 운영 오류, API 계약, 최종 체크리스트를 따로 보지 않고 같은 사건의 evidence path로 묶습니다.

실습 상황은 `high_risk` 비율이 상승했고 릴리스 회의 전에 승인 또는 보류 의견을 내야 하는 상태입니다. QA 담당자는 근거 없이 승인하지 않고, 원인 후보, owner, next action, 재평가 조건을 함께 제시해야 합니다.

| 판단 축 | 확인할 증거 | 가능한 action |
| --- | --- | --- |
| 입력 분포 | feature 평균 변화, shifted 여부, `drift_report.md` | 데이터 출처와 전처리 변경 확인 |
| score/prediction | 평균 score, `high_risk` rate, threshold | threshold와 모델 버전 비교 |
| 운영 상태 | 오류율, latency, validation failure, request trace | API 계약과 client 입력 확인 |
| release gate | metric 기준, 운영 기준, live evidence, checklist | 승인, 조건부 보류, 재평가 |

최종 산출물은 체크리스트 표와 보고서 문장입니다. 숫자를 계산하는 것보다, 그 숫자가 어떤 판단과 재평가 조건으로 이어지는지 설명하는 것이 핵심입니다.

## 1. 실행 환경과 evidence path 고정

### 1-1. 최종 판단에 사용할 artifact 범위 확인

이 셀의 판단은 최종 release 의견이 어떤 evidence file에 근거하는지 고정하는 것입니다. 5장은 새 도구를 배우는 장이 아니라, 앞 장에서 만든 산출물을 감사 추적 가능한 판단으로 묶는 장입니다.

같은 숫자라도 출처가 다르면 의미가 달라집니다. `model_test_eval.json`은 test 평가 기준이고, `drift_report.md`와 운영 로그는 current batch 관측이며, `release_approval.md`는 approval rule을 적용한 제출용 판단입니다.

이 단계에서는 아직 승인/보류를 판단하지 않습니다. 먼저 데이터 계보와 artifact 경계를 표로 남깁니다.

In [ ]:
from __future__ import annotations

import json

import utils as runtime

prepared = await runtime.prepare_notebook()
pd = prepared.pandas
lite = prepared.aiq_lite

for name in runtime.LITE_NAMES:
    globals()[name] = getattr(lite, name)

artifact_contract = pd.DataFrame(
    [
        {"stage": "data_quality", "artifact": "chapter_01_quality_report.md", "qa_boundary": "평가 가능성 확인이며 운영 current 정상으로 확대하지 않습니다."},
        {"stage": "model_quality", "artifact": "model_test_eval.json", "qa_boundary": "선택된 모델과 threshold의 test 평가로 한정합니다."},
        {"stage": "validation_comparison", "artifact": "validation_degradation_comparison.json", "qa_boundary": "validation 품질 저하 비교이며 운영 root cause 확정이 아닙니다."},
        {"stage": "operational_current", "artifact": "drift_report.md, quality_issue_trace.md", "qa_boundary": "current batch 입력 구성 변화와 운영 검증 실패 후보입니다."},
        {"stage": "release_gate", "artifact": "release_approval.md, ai_qa_checklist.md", "qa_boundary": "조건부 보류와 재평가 조건을 evidence path로 남깁니다."},
    ]
)
runtime_contract = pd.DataFrame(
    [
        {"item": "package_module", "value": lite.__name__, "qa_use": "Lite 또는 local helper 기준을 확인합니다."},
        {"item": "notebook_role", "value": "final_release_evidence_workbook", "qa_use": "앞 장 evidence를 최종 판단으로 연결합니다."},
        {"item": "decision_pressure", "value": "release meeting before approval", "qa_use": "승인/보류 의견과 재평가 조건이 필요합니다."},
    ]
)

display(runtime_contract)
display(artifact_contract)

이 출력에서 확인할 핵심은 최종 판단의 출처를 섞지 않는 것입니다. 운영 current에서 보인 `high_risk` 증가를 test 성능 저하로 쓰면 안 되고, validation 비교를 운영 원인 확정으로 쓰면 안 됩니다.

## 2. 입력 데이터 분포 변화 확인

### 2-1. 기준 요청과 current 요청 data brief

이 셀의 판단은 current batch 입력이 operational baseline과 어떻게 다른지 확인하는 것입니다. 한 row는 serving request sample 하나이며, 이 데이터는 정답 label 평가가 아니라 운영 입력 feature 변화 확인에 사용합니다.

입력 분포 변화는 모델 문제의 확정 증거가 아닙니다. 하지만 score와 prediction 변화가 함께 나타나면 원인 후보로 강해집니다.

먼저 source, row grain, row count, feature count를 고정한 뒤 feature 평균 변화를 확인합니다.

In [ ]:
fallback_requests = sample_vital_signs().head(120)
baseline_requests, baseline_source = load_csv_or_sample(
    "data/serving_requests_valid.csv",
    fallback_requests,
    nrows=None,
)
current_requests, current_source = load_csv_or_sample(
    "data/serving_requests_current.csv",
    fallback_requests,
    nrows=None,
)
request_data_brief = pd.DataFrame(
    [
        {"dataset": "serving_requests_valid", "source": baseline_source, "row_grain": "one serving request sample", "row_count": len(baseline_requests), "feature_count": len(FEATURE_COLUMNS)},
        {"dataset": "serving_requests_current", "source": current_source, "row_grain": "one serving request sample", "row_count": len(current_requests), "feature_count": len(FEATURE_COLUMNS)},
    ]
)
feature_comparison = compare_input_distribution(baseline_requests, current_requests, FEATURE_COLUMNS)
shifted_features = feature_comparison.loc[feature_comparison["shifted"], "feature"].tolist()

display(request_data_brief)
display(feature_comparison.sort_values(["shifted", "feature"], ascending=[False, True]))

이 출력에서 확인할 핵심은 shifted feature가 있는지와 그 방향입니다. 입력 분포 변화가 보이면 “모델이 바뀌었다”가 아니라 “입력 조건이 baseline과 달라졌으므로 동일 기준 평가가 필요한 상태”라고 표현합니다.

feature 변화가 있어도 바로 drift 확정이라고 쓰지 않습니다. current batch 기간, 상위 데이터 수집 경로, 배포 변경 여부를 함께 확인해야 합니다.

### 2-2. drift report artifact와 notebook 계산 비교

이 셀의 판단은 notebook에서 계산한 feature 변화가 prepared `drift_report.md`와 같은 판단 흐름에 있는지 확인하는 것입니다. 최종 보고서는 즉석 계산보다 제출용 artifact를 근거로 삼아야 합니다.

artifact를 읽는 목적은 문장을 복사하는 것이 아니라, 어떤 evidence path가 최종 체크리스트와 연결되는지 확인하는 것입니다.

이 출력의 판단은 drift report가 존재하고, notebook의 shifted feature 후보와 같은 종류의 evidence를 담고 있는지 확인하는 것입니다.

In [ ]:
drift_report_text = runtime.read_text_artifact("artifacts/reports/drift_report.md")
drift_report_check = pd.DataFrame(
    [
        {"check": "drift_report_exists", "observed": bool(drift_report_text.strip()), "qa_use": "제출용 입력 변화 산출물입니다."},
        {"check": "shifted_features_from_notebook", "observed": ", ".join(shifted_features) or "none", "qa_use": "notebook 계산에서 확인한 입력 변화 후보입니다."},
        {"check": "mentions_heart_rate", "observed": "heart_rate" in drift_report_text, "qa_use": "주요 feature 변화가 report에 남았는지 확인합니다."},
        {"check": "mentions_oxygen_saturation", "observed": "oxygen_saturation" in drift_report_text, "qa_use": "주요 feature 변화가 report에 남았는지 확인합니다."},
    ]
)
print("\\n".join(drift_report_text.splitlines()[:12]))
display(drift_report_check)

이 출력에서 확인할 핵심은 notebook 계산과 제출 artifact가 같은 판단을 가리키는지입니다. 직접 계산하지 않은 경우에도 prepared artifact에서 확인한 값이라고 보고서에 명시하면 됩니다.

## 3. score와 prediction 분포 비교

### 3-1. 운영 로그에서 score와 `high_risk` 비율 변화 확인

이 셀의 판단은 입력 변화가 output 변화와 함께 나타나는지 확인하는 것입니다. score는 prediction 이전의 연속값이고, prediction은 threshold를 적용한 최종 class입니다.

운영 로그는 정답 label이 없는 current 관측일 수 있습니다. 따라서 Precision/Recall을 바로 말하지 않고, score 평균, `high_risk` rate, validation failure, latency 같은 proxy signal을 봅니다.

이 단계의 출력은 입력 분포 변화와 연결해 읽습니다. feature 변화, score 변화, prediction 변화가 같은 방향이면 데이터 조건 변화와 모델 출력 변화가 연결된 후보로 기록합니다.

In [ ]:
def read_jsonl_artifact(path: str, max_events: int | None = 120) -> list[dict[str, object]]:
    text = runtime.read_text_artifact(path)
    events = [json.loads(line) for line in text.splitlines() if line.strip()]
    return events[:max_events] if max_events is not None else events


baseline_events = read_jsonl_artifact("artifacts/logs/chapter_04_normal_events.jsonl")
current_events = read_jsonl_artifact("artifacts/logs/chapter_04_anomaly_events.jsonl")
baseline_snapshot = quality_snapshot(baseline_events)
current_snapshot = quality_snapshot(current_events)
score_comparison = score_distribution_comparison(baseline_events, current_events)
score_comparison_table = pd.DataFrame([score_comparison])
snapshot_table = pd.DataFrame(
    [
        {"signal": key, "baseline": baseline_snapshot.get(key), "current": current_snapshot.get(key)}
        for key in sorted(set(baseline_snapshot) | set(current_snapshot))
    ]
)

display(snapshot_table)
display(score_comparison_table)

이 출력에서 확인할 핵심은 `high_risk` 비율 상승만 단독으로 쓰지 않는 것입니다. 평균 score 변화, threshold 유지 여부, validation failure, latency를 함께 봐야 운영자가 같은 결론을 재현할 수 있습니다.

### 3-2. 원인 후보를 owner와 next action으로 정리

이 셀의 판단은 입력 변화, prediction shift, API validation, service latency를 각각 다른 owner와 next action으로 분리하는 것입니다. 후보를 하나로 합치면 후속 조치가 흐려집니다.

`trace_candidates`는 앞 셀의 feature comparison, score comparison, operational delta를 받아 후보 표를 만듭니다. 이 helper는 결론을 자동으로 승인하지 않고, 보고서에 필요한 owner와 next action을 정리하는 역할입니다.

이 표는 `quality_issue_trace.md`와 함께 최종 체크리스트에 들어갈 evidence입니다.

In [ ]:
quality_report = compare_snapshots(baseline_snapshot, current_snapshot)
candidates = trace_candidates(feature_comparison, score_comparison, quality_report, current_events)
quality_issue_trace_text = runtime.read_text_artifact("artifacts/reports/quality_issue_trace.md")
quality_delta_table = pd.DataFrame(
    [
        {"signal": "error_rate_delta", "observed": quality_report["error_rate_delta"], "candidate": "api_validation"},
        {"signal": "latency_delta_ms", "observed": quality_report["latency_delta_ms"], "candidate": "service_latency"},
        {"signal": "high_risk_rate_delta", "observed": quality_report["high_risk_rate_delta"], "candidate": "prediction_shift"},
        {"signal": "average_score_delta", "observed": quality_report["average_score_delta"], "candidate": "score_distribution_shift"},
    ]
)

display(quality_delta_table)
display(candidates)
print("\\n".join(quality_issue_trace_text.splitlines()[:10]))

이 출력에서 확인할 핵심은 원인 후보가 조치 가능한 단위로 바뀌었다는 점입니다. input case mix는 Data Engineering, API validation은 Client Integration, latency는 Platform/MLOps처럼 owner가 나뉩니다.

## 4. release gate와 승인/보류 판단

### 4-1. release regression data와 label basis 확인

이 셀의 판단은 release gate 판단에 들어가는 데이터의 label basis를 먼저 확인하는 것입니다. 모델 품질 지표를 계산하거나 approval report를 읽기 전에 label 허용값과 class support를 확인해야 합니다.

`release_regression_cases.csv`는 최종 승인 판단 예제에 쓰는 작은 regression case 묶음입니다. 한 row는 회귀 테스트 sample이고, `label`은 평가 기준 정답입니다.

이 단계에서는 모델 개선을 목표로 하지 않습니다. label basis가 지표 해석에 충분한지, 그리고 제출용 `label_basis_check.md`가 같은 근거를 남기는지 확인합니다.

In [ ]:
release_cases, release_case_source = load_csv_or_sample(
    "data/release_regression_cases.csv",
    sample_vital_signs(),
    nrows=None,
)
release_label_distribution = (
    release_cases["label"]
    .value_counts(dropna=False)
    .rename("count")
    .reset_index()
    .rename(columns={"index": "label"})
    .assign(ratio_pct=lambda table: (table["count"] / len(release_cases) * 100).round(2))
)
label_basis_text = runtime.read_text_artifact("artifacts/reports/label_basis_check.md")
release_data_brief = pd.DataFrame(
    [
        {"check": "release_case_source", "observed": release_case_source, "qa_use": "release gate 예제 데이터 출처입니다."},
        {"check": "row_grain", "observed": "one release regression sample", "qa_use": "지표 분모를 설명합니다."},
        {"check": "row_count", "observed": len(release_cases), "qa_use": "label support와 metric 계산 기준입니다."},
        {"check": "label_values", "observed": ", ".join(sorted(release_cases["label"].dropna().astype(str).unique())), "qa_use": "허용 label 유지 여부입니다."},
    ]
)

display(release_data_brief)
display(release_label_distribution)
print("\\n".join(label_basis_text.splitlines()[:12]))

이 출력에서 확인할 핵심은 release gate의 지표가 어떤 label support에서 나온 것인지 설명할 수 있어야 한다는 점입니다. label basis가 불명확하면 `inconclusive` 또는 보류 판단을 남기는 것이 안전합니다.

### 4-2. approval rule 결과와 unresolved risk 확인

이 셀의 판단은 승인 여부보다 실패 기준과 미검증 리스크를 먼저 읽는 것입니다. `approved=False` 자체보다 어떤 기준이 실패했고 어떤 evidence가 없는지가 중요합니다.

이 notebook은 최종 판단의 source of truth를 `release_approval.md`로 둡니다. prepared API contract가 통과했더라도 live deployment evidence가 없으면 운영 배포 검증 완료라고 말하지 않습니다.

이 표의 판단은 제출용 approval report가 어떤 check를 통과하거나 실패했는지 확인하는 것입니다. Notebook 계산과 보고서 artifact가 다른 값을 만들면 audit에서 방어되지 않으므로, 이 셀은 report artifact를 직접 파싱합니다.

In [ ]:
release_approval_text = runtime.read_text_artifact("artifacts/reports/release_approval.md")


def parse_markdown_table(text: str, header_start: str) -> pd.DataFrame:
    lines = text.splitlines()
    start = next(index for index, line in enumerate(lines) if line.startswith(header_start))
    rows: list[list[str]] = []
    for line in lines[start:]:
        if not line.startswith("|"):
            if rows:
                break
            continue
        cells = [cell.strip() for cell in line.strip("|").split("|")]
        rows.append(cells)
    header = rows[0]
    body = [row for row in rows[2:] if len(row) == len(header)]
    return pd.DataFrame(body, columns=header)


decision_summary_table = parse_markdown_table(
    release_approval_text,
    "| recommendation | approved | failed_checks | unresolved_risks |",
)
release_checks = parse_markdown_table(
    release_approval_text,
    "| check | observed | criterion | result |",
)
unresolved_risk_table = parse_markdown_table(
    release_approval_text,
    "| area | status | evidence | owner | next_action |",
)
release_decision = {
    "recommendation": decision_summary_table.loc[0, "recommendation"],
    "approved": decision_summary_table.loc[0, "approved"] == "True",
    "failed_checks": [
        item.strip()
        for item in decision_summary_table.loc[0, "failed_checks"].split(",")
        if item.strip() and item.strip() != "-"
    ],
    "unresolved_risks": [
        item.strip()
        for item in decision_summary_table.loc[0, "unresolved_risks"].split(",")
        if item.strip() and item.strip() != "-"
    ],
}
release_report_consistency = pd.DataFrame(
    [
        {
            "check": "precision_pass_in_report",
            "observed": release_checks.loc[release_checks["check"].eq("precision"), "result"].iloc[0],
            "qa_use": "정밀도는 제출용 report에서 pass로 남아야 합니다.",
        },
        {
            "check": "failed_checks_from_report",
            "observed": ", ".join(release_decision["failed_checks"]),
            "qa_use": "최종 보고 문장은 이 실패 기준만 인용합니다.",
        },
        {
            "check": "unresolved_risks_from_report",
            "observed": ", ".join(release_decision["unresolved_risks"]),
            "qa_use": "live evidence 부족을 숨기지 않습니다.",
        },
    ]
)

display(decision_summary_table)
display(release_checks)
display(unresolved_risk_table)
display(release_report_consistency)
print("\\n".join(release_approval_text.splitlines()[:18]))

이 출력에서 확인할 핵심은 조건부 보류가 실패가 아니라 방어 가능한 품질 판단이라는 점입니다. Recall 또는 error rate 기준이 실패하고 live evidence가 없으면 승인 대신 재평가 조건을 남깁니다.

prepared report가 있으므로 최종 보고서에는 “notebook helper로 확인한 rule 흐름”과 “prepared artifact에서 확인한 제출 판단”을 구분해서 적습니다.

## 5. 최종 AI QA 체크리스트 작성

### 5-1. 체크리스트 상태와 evidence path 확인

이 셀의 판단은 체크리스트가 단순 완료 목록이 아니라 release 판단 문서인지 확인하는 것입니다. 각 항목에는 상태, evidence, QA 코멘트, 담당, 다음 조치가 있어야 합니다.

`ai_qa_checklist.md`는 이번 사건 제출용 점검본이고, `ai_qa_checklist_template.md`는 반복 사용 템플릿입니다. 제출본은 `pass`, `fail`, `unverified`, `hold`를 구분해야 합니다.

이 표의 판단은 notebook에서 만든 후보와 prepared checklist의 핵심 상태가 연결되는지 확인하는 것입니다.

In [ ]:
checklist_text = runtime.read_text_artifact("artifacts/reports/ai_qa_checklist.md")
checklist = pd.DataFrame(
    [
        {"item": "input distribution", "status": "blocked" if bool(feature_comparison["shifted"].any()) else "ready", "owner": "Data Engineering", "evidence": "drift_report.md"},
        {"item": "score and prediction distribution", "status": "blocked" if score_comparison["high_risk_rate_delta"] > 0.15 else "ready", "owner": "ML Engineering", "evidence": "drift_report.md"},
        {"item": "api validation failures", "status": "blocked" if quality_report["error_rate_delta"] > 0.03 else "ready", "owner": "Client Integration", "evidence": "quality_issue_trace.md"},
        {"item": "service latency", "status": "blocked" if quality_report["latency_delta_ms"] > 100 else "ready", "owner": "Platform/MLOps", "evidence": "quality_issue_trace.md"},
        {"item": "live deployment evidence", "status": "blocked", "owner": "Platform/MLOps", "evidence": "release_approval.md"},
        {"item": "final decision", "status": "hold" if not release_decision["approved"] else "ready", "owner": "QA Lead", "evidence": "release_approval.md"},
    ]
)
checklist_artifact_check = pd.DataFrame(
    [
        {"check": "release_ready_state", "observed": "릴리스 준비 상태: blocked" in checklist_text, "qa_use": "제출본이 blocked 상태를 명시하는지 확인합니다."},
        {"check": "conditional_hold", "observed": "conditional_hold" in checklist_text, "qa_use": "조건부 보류 판단이 남았는지 확인합니다."},
        {"check": "live_deployment_unverified", "observed": "live_deployment" in checklist_text and "unverified" in checklist_text, "qa_use": "미검증 live evidence를 숨기지 않았는지 확인합니다."},
        {"check": "owner_next_action", "observed": "다음 조치" in checklist_text, "qa_use": "owner별 action이 있는지 확인합니다."},
    ]
)
release_ready = not checklist["status"].isin(["blocked", "hold"]).any()

display(checklist)
display(checklist_artifact_check)
print("release_ready", release_ready)

이 출력에서 확인할 핵심은 checklist가 `blocked`와 `hold`를 명확히 표시한다는 점입니다. 누락된 live deployment evidence를 숨기면 release 이후 문제가 생겼을 때 QA 판단을 방어하기 어렵습니다.

### 5-2. 최종 보고 문장과 재평가 조건 생성

마지막 판단은 승인/보류 의견을 하나의 보고 문장으로 정리하는 것입니다. 문장에는 증상, 근거, 원인 후보, owner, release recommendation, 재평가 조건이 함께 있어야 합니다.

“승인합니다” 또는 “보류합니다”만으로 끝내면 실무 산출물로 부족합니다. 어떤 기준을 통과했고, 어떤 기준을 실패했으며, 실패 항목을 해결하기 위해 누가 무엇을 확인해야 하는지 남겨야 합니다.

이 출력의 판단은 회의에서 사용할 수 있는 보고 문장 초안이 evidence, owner, 재평가 조건을 포함하는지 확인하는 것입니다.

In [ ]:
owner_actions = candidates.loc[:, ["candidate", "owner", "next_action"]].to_dict("records")
report_sentence = (
    f"릴리스 판단은 {release_decision['recommendation']}입니다. "
    f"current 관측에서 high_risk_rate_delta={score_comparison['high_risk_rate_delta']:.4f}, "
    f"average_score_delta={score_comparison['average_score_delta']:.4f}, "
    f"error_rate_delta={quality_report['error_rate_delta']}, latency_delta_ms={quality_report['latency_delta_ms']}가 확인되었습니다. "
    f"입력 변화 후보는 {', '.join(shifted_features) if shifted_features else 'none'}이며, "
    f"failed_checks={release_decision['failed_checks']}와 unresolved_risks={release_decision['unresolved_risks']}가 남아 있습니다. "
    "따라서 owner별 next action 완료 후 같은 approval rule과 live deployment evidence로 재평가합니다."
)
final_evidence_packet = pd.DataFrame(
    [
        {"evidence": "input_shift", "observed": ", ".join(shifted_features) or "none", "owner": "Data Engineering", "next_action": "데이터 수집 경로와 upstream feed 변경 여부 확인"},
        {"evidence": "prediction_shift", "observed": f"high_risk_rate_delta={score_comparison['high_risk_rate_delta']:.4f}", "owner": "ML Engineering", "next_action": "threshold와 score distribution 재확인"},
        {"evidence": "api_validation", "observed": f"error_rate_delta={quality_report['error_rate_delta']}", "owner": "Client Integration", "next_action": "failed_field와 client payload 변경 확인"},
        {"evidence": "release_gate", "observed": f"recommendation={release_decision['recommendation']}, approved={release_decision['approved']}", "owner": "QA Lead", "next_action": "같은 approval rule로 재평가"},
    ]
)

display(final_evidence_packet)
print(report_sentence)

### 5-3. 실패 시 확인 포인트

데이터 파일을 읽지 못하면 먼저 `data/serving_requests_valid.csv`, `data/serving_requests_current.csv`, `data/release_regression_cases.csv`가 Lite files 또는 로컬 repo에 복사되었는지 확인합니다. fallback sample로 실행된 경우 보고서에 sample 기준이라고 적어야 합니다.

최종 report artifact를 읽지 못하면 `artifacts/reports/drift_report.md`, `quality_issue_trace.md`, `release_approval.md`, `ai_qa_checklist.md`가 준비되었는지 확인합니다. 로컬에서는 `uv run --group lab python labs/ch05_qa_strategy/build_qa_artifacts.py`로 재생성합니다.

`release_ready=True`가 나오더라도 live deployment evidence가 미검증이면 운영 배포 준비 완료라고 쓰지 않습니다. `/health`, `/predict`, Pod readiness, 응답 `model_version`, `threshold` 증거를 별도로 확인해야 합니다.

최종 문장은 원인 확정 문장으로 쓰지 않습니다. 입력 변화, validation failure, latency, prediction shift를 후보로 분리하고 owner별 재평가 조건을 남깁니다.